In [0]:
%run ../configs/config

In [0]:
orders = f'{bronze_schema}.orders'
orders_df = spark.table(orders)

In [0]:
display(orders_df)


bank of the recipient


account_to
account of the recipient


amount
amount debited from order account


k_symbol
characterization of the payment

In [0]:
orders_renamed= orders_df.withColumnsRenamed({'bank_to':'recipient_bank','account_to':'recipient_account',
                               'amount':'order_amount', 'k_symbol':'payment_type'})

SIPO Household Payment
UVER Loan Payment  
(empty) Unknown
POJISTNE Insurance Payment
LEASING Leasing Payment

In [0]:
orders_format = (orders_renamed
                 .withColumn('payment_type',
                              F.when(F.col('payment_type') == 'SIPO', 'Household Payment')
                              .when(F.col('payment_type') == 'UVER', 'Loan Payment')
                              .when(F.col('payment_type') == ' ', 'Unknown')
                              .when(F.col('payment_type') == 'POJISTNE', 'Insurance Payment')
                              .when(F.col('payment_type') == 'LEASING', 'Leasing Payment')
                              .otherwise('Unknown')
                              )
                 .dropDuplicates()
                 .filter(F.col('order_id').isNotNull())
                 
        
        )

In [0]:
display(orders_format)

In [0]:
(
    orders_format
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(f'{silver_schema}.orders')
)